# Embedding Pipeline Notebook

Notebook này tự động hóa pipeline từ data ingestion -> chunking -> compare strategy -> embedding -> thống kê embedding.

Thiết kế ưu tiên chạy CPU: bắt đầu bằng subset nhỏ qua `ARTICLE_LIMIT`, quan sát tiến độ embed bằng `tqdm` từ CLI, sau đó mới tăng limit hoặc chạy full corpus.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import time

import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)

## 1. Cấu hình

- `RAW_CSV_PATH`: CSV đầu vào cho `src.data_ingestion.cli`.
- `ARTICLE_LIMIT`: số article đưa vào chunking/embedding. Đặt `None` để chạy full corpus.
- `STRATEGIES`: các chiến thuật chunking cần embed.
- `BATCH_SIZE`: CPU nên bắt đầu thấp, ví dụ 4 hoặc 8.

In [ ]:
# Chọn một CSV đang có trong repo. Có thể đổi sang validation_new.csv/test_new.csv hoặc CSV tổng của bạn.
RAW_CSV_PATH = PROJECT_ROOT / "Dataset" / "Create_QA_Vietonline" / "VietOnlineNews" / "train_new.csv"

PROCESSED_JSONL = PROJECT_ROOT / "src" / "embed" / "output" / "data" / "vieonline_news_clean.jsonl"
REVIEW_JSONL = PROJECT_ROOT / "src" / "embed" / "output" / "data" / "vieonline_news_human_review.jsonl"
CHUNK_OUTPUT_DIR = PROJECT_ROOT / "src" / "embed" / "output" / "chunks"
DENSE_OUTPUT_ROOT = PROJECT_ROOT / "src" / "embed" / "output" / "dense"
SAMPLE_REPORT = PROJECT_ROOT / "src" / "embed" / "output" / "embedding_strategy_samples.md"

# CPU-friendly default. Đặt None để chạy toàn bộ file sau khi đã smoke test.
ARTICLE_LIMIT = 200
STRATEGIES = ["token", "llamaindex", "structured"]  # thêm "langchain_recursive" nếu muốn so sánh đủ 4 strategy

CHUNK_SIZE = 400
OVERLAP = 80
SMALL_ARTICLE_CHARS = 1000
BATCH_SIZE = 4
MODEL_NAME = "intfloat/multilingual-e5-large"
SAMPLE_SIZE = 5

for path in [PROCESSED_JSONL.parent, CHUNK_OUTPUT_DIR, DENSE_OUTPUT_ROOT, SAMPLE_REPORT.parent]:
    path.mkdir(parents=True, exist_ok=True)

config = {
    "raw_csv": str(RAW_CSV_PATH),
    "processed_jsonl": str(PROCESSED_JSONL),
    "chunk_output_dir": str(CHUNK_OUTPUT_DIR),
    "dense_output_root": str(DENSE_OUTPUT_ROOT),
    "article_limit": ARTICLE_LIMIT,
    "strategies": STRATEGIES,
    "chunk_size": CHUNK_SIZE,
    "overlap": OVERLAP,
    "batch_size": BATCH_SIZE,
    "model_name": MODEL_NAME,
}
display(config)

## 2. Helper chạy CLI có streaming output

Các lệnh bên dưới dùng `sys.executable -m ...` để chạy đúng Python environment của notebook. Output được stream trực tiếp để thấy progress bar của chunking và embedding.

In [ ]:
def run_cli(args, cwd=PROJECT_ROOT):
    args = [str(item) for item in args]
    printable = " ".join(args)
    print(f"\n$ {printable}\n")
    started = time.perf_counter()
    process = subprocess.Popen(
        args,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        lines.append(line)
    return_code = process.wait()
    elapsed = time.perf_counter() - started
    print(f"\n[exit={return_code}] elapsed={elapsed:.2f}s")
    if return_code != 0:
        raise RuntimeError(f"Command failed: {printable}")
    return "".join(lines)

## 3. Data ingestion

In [ ]:
ingest_cmd = [
    sys.executable, "-m", "src.data_ingestion.cli",
    "--input", RAW_CSV_PATH,
    "--output", PROCESSED_JSONL,
    "--review-output", REVIEW_JSONL,
]
run_cli(ingest_cmd)

## 4. Chunking theo strategy

Bước này dùng CLI `src.chunking.cli`. Nếu chạy CPU, giữ `ARTICLE_LIMIT` nhỏ trước để giảm thời gian embed sau đó.

In [ ]:
chunk_cmd = [
    sys.executable, "-m", "src.chunking.cli",
    "--input", PROCESSED_JSONL,
    "--output-dir", CHUNK_OUTPUT_DIR,
    "--strategies", *STRATEGIES,
    "--chunk-size", CHUNK_SIZE,
    "--overlap", OVERLAP,
    "--small-article-chars", SMALL_ARTICLE_CHARS,
]
if ARTICLE_LIMIT is not None:
    chunk_cmd += ["--limit", ARTICLE_LIMIT]
run_cli(chunk_cmd)

## 5. Xem mẫu khác biệt input embedding giữa các strategy

In [ ]:
chunk_files = [CHUNK_OUTPUT_DIR / f"vieonline_news_chunks_{strategy}.jsonl" for strategy in STRATEGIES]
compare_cmd = [
    sys.executable, "-m", "src.embed.compare_embedding_inputs",
    "--inputs", *chunk_files,
    "--sample-articles", 3,
    "--output", SAMPLE_REPORT,
]
run_cli(compare_cmd)
display(Markdown(SAMPLE_REPORT.read_text(encoding="utf-8")))

## 6. Embed từng strategy và quan sát tiến độ

Cell này là bước chậm nhất trên CPU. `tqdm` sẽ hiển thị tiến độ theo batch. Nếu quá chậm, giảm `ARTICLE_LIMIT`, giảm số `STRATEGIES`, hoặc giữ `BATCH_SIZE=4`.

In [ ]:
for strategy, chunk_file in zip(STRATEGIES, chunk_files):
    output_dir = DENSE_OUTPUT_ROOT / strategy
    embed_cmd = [
        sys.executable, "-m", "src.embed.embed_chunks",
        "--input", chunk_file,
        "--output-dir", output_dir,
        "--model", MODEL_NAME,
        "--batch-size", BATCH_SIZE,
        "--sample-size", SAMPLE_SIZE,
        "--show-samples",
    ]
    run_cli(embed_cmd)

## 7. Bảng thống kê embedding

In [ ]:
def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

rows = []
for strategy in STRATEGIES:
    strategy_dir = DENSE_OUTPUT_ROOT / strategy
    stats_path = strategy_dir / "embedding_stats.json"
    manifest_path = strategy_dir / "manifest.json"
    if not stats_path.exists():
        print(f"Missing stats for {strategy}: {stats_path}")
        continue
    stats = load_json(stats_path)
    manifest = load_json(manifest_path)
    rows.append({
        "strategy": strategy,
        "chunks": stats["chunks_embedded"],
        "dimension": stats["embedding_dimension"],
        "batch_size": stats["batch_size"],
        "elapsed_seconds": stats["elapsed_seconds"],
        "chunks_per_second": stats["chunks_per_second"],
        "avg_tokens": stats["token_stats"]["avg"],
        "max_tokens": stats["token_stats"]["max"],
        "avg_chars": stats["char_stats"]["avg"],
        "max_chars": stats["char_stats"]["max"],
        "embeddings_path": manifest["embeddings_path"],
        "stats_path": str(stats_path),
    })

embed_stats_df = pd.DataFrame(rows).sort_values("strategy")
display(embed_stats_df)

summary_csv = PROJECT_ROOT / "src" / "embed" / "output" / "embedding_stats_summary.csv"
embed_stats_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
print("Saved:", summary_csv)

## 8. Debug chunk dài nhất và mẫu đã embed

In [ ]:
for strategy in STRATEGIES:
    strategy_dir = DENSE_OUTPUT_ROOT / strategy
    stats_path = strategy_dir / "embedding_stats.json"
    samples_path = strategy_dir / "debug_samples.jsonl"
    if not stats_path.exists():
        continue
    stats = load_json(stats_path)
    display(Markdown(f"### {strategy}: longest chunks"))
    display(pd.DataFrame(stats["longest_chunks"]))
    if samples_path.exists():
        samples = [json.loads(line) for line in samples_path.read_text(encoding="utf-8").splitlines() if line.strip()]
        display(Markdown(f"### {strategy}: debug samples"))
        display(pd.DataFrame(samples))

## 9. Search thử trên một strategy đã embed

In [ ]:
QUERY = "Tin tức về công nghệ AI tại Việt Nam"
SEARCH_STRATEGY = STRATEGIES[0]

search_cmd = [
    sys.executable, "-m", "src.embed.dense_search",
    "--index-dir", DENSE_OUTPUT_ROOT / SEARCH_STRATEGY,
    "--query", QUERY,
    "--top-k", 5,
    "--model", MODEL_NAME,
]
run_cli(search_cmd)